In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import shutil
import string
from sklearn.metrics import classification_report
import tensorflow as tf
from keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras import losses
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.layers import Dense, Input, Dropout, Bidirectional, LSTM, Embedding, BatchNormalization,  Reshape, Conv2D, MaxPool2D, concatenate, Flatten, Activation
import torch
import numpy as np
from transformers import BertTokenizer, BertModel,RobertaTokenizer, RobertaModel,AutoTokenizer, AutoModel
import ast

In [ ]:
# Verificar si la GPU está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
archivo_3 = '/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsBERT/DataProgramsandDescriptions-CatRetroalimentacion5000.xlsx'
train_full = pd.read_excel(archivo_3)

# AST spliter D. Gries form

In [ ]:
class WhileLoopFinder(ast.NodeVisitor):
    def __init__(self, source_code):
        self.source_code = source_code.splitlines()
        self.functions_with_while = []
        self.current_function = None

    def visit_FunctionDef(self, node):
        original_current_function = self.current_function
        self.current_function = {
            "function_name": node.name,
            "has_while_loop": False,
            "pre_while_code_lines": [],
            "while_loops": []
        }

        function_start_line = node.lineno - 1
        function_end_line = (node.end_lineno if hasattr(node, 'end_lineno') else len(self.source_code))
        function_lines = self.source_code[function_start_line:function_end_line]

        found_while = False
        for stmt in node.body:
            if isinstance(stmt, ast.While):
                self.current_function["has_while_loop"] = True
                found_while = True
                try:
                    while_condition = ast.unparse(stmt.test).strip()
                    print("OK condition")
                except AttributeError:
                    print("Error al obtener la condición del while")
                try:
                    while_body_code = ast.unparse(ast.Module(body=stmt.body, type_ignores=[])).strip()
                    print("Ok body")
                except AttributeError:
                    print("Error al obtener el código del cuerpo del while")

                self.current_function["while_loops"].append({
                    "condition": while_condition,
                    "body_code": while_body_code
                })
            elif not found_while:
                start_line = stmt.lineno - 1
                end_line = (stmt.end_lineno if hasattr(stmt, 'end_lineno') else stmt.lineno)
                self.current_function["pre_while_code_lines"].extend(self.source_code[start_line:end_line])
            self.generic_visit(stmt)

        if self.current_function["has_while_loop"]:
            last_while_end_line = None
            if self.current_function["while_loops"]:
                last_while_node = next((node for node in reversed(node.body) if isinstance(node, ast.While)), None)
                if last_while_node and hasattr(last_while_node, 'end_lineno'):
                    last_while_end_line = last_while_node.end_lineno
            post_while_code_lines = []
            if last_while_end_line:
                function_indent = len(self.source_code[node.lineno - 1]) - len(self.source_code[node.lineno - 1].lstrip())
                for line in self.source_code[last_while_end_line:function_end_line]:
                    if line.startswith(self.source_code[node.lineno - 1][:function_indent] + "    "):
                        post_while_code_lines.append(line[function_indent + 4:])
                    else:
                        post_while_code_lines.append(line[function_indent:])
            self.current_function["post_while_code_lines"] = "\n".join(post_while_code_lines).strip()
            self.current_function["pre_while_code_lines"] = "\n".join(self.current_function["pre_while_code_lines"]).strip()
            self.functions_with_while.append(self.current_function)

        self.current_function = original_current_function



In [ ]:
def DGries_states(solution):
  try:
    # Crear el AST
    tree = ast.parse(solution)
    # Recorrer el AST con nuestro visitante
    finder = WhileLoopFinder(solution)
    finder.visit(tree)
    for func_info in finder.functions_with_while:
      initial_state=func_info["pre_while_code_lines"] if func_info["pre_while_code_lines"] else False
      end_state=""
      transformation_state=""
      for j, wl in enumerate(func_info["while_loops"]):
          end_state=wl["condition"]
          transformation_state=wl["body_code"]

      if not initial_state:
        initial_state=transformation_state
      return initial_state,transformation_state,end_state
  except Exception as error:
    return "Exception:"+str(error),"Exception:"+str(error),"Exception:"+str(error)




# Encoder Description and Code

In [ ]:
class Encoder:
  def __init__(self):
    self.is_loadtokenizers=False



  def tokenize_and_generate_embeddings_descriptions(self,description):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input description and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(description, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.beart_model(**tokens)
    # Extract embeddings for all tokens
    desc_embeddings = outputs.last_hidden_state.cpu().numpy()
    return desc_embeddings

  def tokenize_and_generate_embeddings_codes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.codebeart_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.codebeart_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()

    return code_embeddings


  def tokenize_and_generate_embeddings_graphcodes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.graphcodebert_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.graphcodebert_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()
    return code_embeddings

  def load_beart_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    model_name = "bert-base-uncased"
    self.beart_model = BertModel.from_pretrained(model_name)
    self.beart_tokenizer = BertTokenizer.from_pretrained(model_name)
    self.beart_model.to(device)

  def load_graphcodebert_tokenizer(self):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    self.graphcodebert_tokenizer = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
    self.graphcodebert_model = AutoModel.from_pretrained("microsoft/graphcodebert-base")
    self.graphcodebert_model.to(device)


  def load_codebert_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    self.codebeart_tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    self.codebeart_model = RobertaModel.from_pretrained("microsoft/codebert-base")
    self.codebeart_model.to(device)

  def start_tokenizer(self):
    self.load_beart_tokenizer()
    #self.load_codebert_tokenizer()
    self.load_graphcodebert_tokenizer()
    self.is_loadtokenizers=True




# All characteristics

In [ ]:
def categoricallabelAll(w):
  if w=="['Correct']":
    return 0
  if w=="['Initial state']":
    return 1
  if w=="['Final state']":
    return 2
  if w=="['State transformation']":
    return 3
  if w=="['Initial state', 'Final state']":
    return 4
  if w=="['Initial state', 'State transformation']":
    return 5
  if w=="['Final state', 'State transformation']":
    return 6
  if w=="['Initial state', 'Final state', 'State transformation']":
    return 7
  return 8

category=np.array([
    'Correct',
    'Initial state',
    'Final state',
    'State transformation',
    'Initial state, Final state',
    'Initial state, State transformation',
    'Final state, State transformation',
    'Initial state, Final state, State transformation'

])

# Load Dataset

In [ ]:
train_full.head()

,No.,Problema,Solución,Estado incial,Estado final,Transformación de estado,Etiqueta 1,Etiqueta 2,Realimentación
0,1,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,"result = 1, i = 1",i <= n,"result *= i, i += 1",Correct,['Correct'],NaN
1,2,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,"total = 0, i = 1, n = 100",i <= n,"total += i, i += 1",Correct,['Correct'],NaN
2,3,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,"numbers = [], i = 0, n = 10",i <= n,"print(i), numbers.append(i), i += 1",Correct,['Correct'],NaN
3,4,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,"numbers = [], i = 10, n = 1",i >= n,"print(i), numbers.append(i), i -= 1",Correct,['Correct'],NaN
4,5,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,"i = 2, i = = 0:",i <= num//2,"if num % i == 0:, return False, i += 1",Correct,['Correct'],NaN


In [ ]:
train_full.drop(["No.","Realimentación","Estado incial","Estado final","Transformación de estado"],axis=1,inplace=True)
train_full

,Problema,Solución,Etiqueta 1,Etiqueta 2
0,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,Correct,['Correct']
1,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,Correct,['Correct']
2,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,Correct,['Correct']
3,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,Correct,['Correct']
4,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,Correct,['Correct']
...,...,...,...,...
4995,Write a Python function that returns the sum o...,def sum_of_first_five_numbers():\n number =...,Incorrect,"['Initial state', 'Final state']"
4996,Write a Python function that returns the facto...,def factorial(n):\n result = 0\n i = 0\n...,Incorrect,"['Initial state', 'Final state']"
4997,Write a Python function that prints the number...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']"
4998,Write a Python function to print the numbers f...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']"


In [ ]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Solución'].apply(DGries_states).apply(pd.Series)

Streaming output truncated to the last 5000 lines.
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK c

In [ ]:
y=train_full['Etiqueta 2'].apply(categoricallabelAll)
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6, 7])

In [ ]:
train_full.dropna(inplace=True)

In [ ]:
np.unique(train_full['Etiqueta 2'])

array(["['Correct']", "['Final state', 'State transformation']",
       "['Final state']",
       "['Initial state', 'Final state', 'State transformation']",
       "['Initial state', 'Final state']",
       "['Initial state', 'State transformation']", "['Initial state']",
       "['State transformation']"], dtype=object)

In [ ]:
train_full

,Problema,Solución,Etiqueta 1,Etiqueta 2,Estado incial,Transformación de estado,Estado final
0,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,Correct,['Correct'],result = 1\n i = 1,result *= i\ni += 1,i <= n
1,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,Correct,['Correct'],total = 0\n i = 1\n n = 100,total += i\ni += 1,i <= n
2,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,Correct,['Correct'],numbers = []\n i = 0\n n = 10,print(i)\nnumbers.append(i)\ni += 1,i <= n
3,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,Correct,['Correct'],numbers = []\n i = 10\n n = 1,print(i)\nnumbers.append(i)\ni -= 1,i >= n
4,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,Correct,['Correct'],if num <= 1:\n return False\n i = 2,if num % i == 0:\n return False\ni += 1,i <= num // 2
...,...,...,...,...,...,...,...
4995,Write a Python function that returns the sum o...,def sum_of_first_five_numbers():\n number =...,Incorrect,"['Initial state', 'Final state']",number = 1\n total_sum = 0\n count = 1,total_sum = total_sum + number\ncount = count - 1,count < 5
4996,Write a Python function that returns the facto...,def factorial(n):\n result = 0\n i = 0\n...,Incorrect,"['Initial state', 'Final state']",result = 0\n i = 0,result *= i\ni += 1,i < n
4997,Write a Python function that prints the number...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']",numbers = []\n n = 1,print(n)\nnumbers.append(n)\nn += 1,n <= 11
4998,Write a Python function to print the numbers f...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']",numbers = []\n n = 9,print(n)\nnumbers.append(n)\nn -= 1,n >= 0


In [ ]:
encoder=Encoder()
encoder.start_tokenizer()

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [ ]:
%%time
problem=train_full['Problema'].apply(encoder.tokenize_and_generate_embeddings_descriptions).to_numpy()
code=train_full['Solución'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
startstate = train_full['Estado incial'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
finalstate = train_full['Estado final'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
transstate = train_full['Transformación de estado'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()

CPU times: user 3min 16s, sys: 1.3 s, total: 3min 18s
Wall time: 3min 18s


In [ ]:
problem.shape,code.shape,startstate.shape,finalstate.shape,transstate.shape

((4919,), (4919,), (4919,), (4919,), (4919,))

In [ ]:
print(startstate.shape)
print(startstate[1262].shape)


(4919,)
(1, 17, 768)


In [ ]:
Xp=np.array([sentence[0].mean(axis=0) for sentence in problem])
Xcode=np.array([sentence[0].mean(axis=0) for sentence in code])
Xs=np.array([sentence[0].mean(axis=0) for sentence in startstate])
Xf=np.array([sentence[0].mean(axis=0) for sentence in finalstate])
Xt=np.array([sentence[0].mean(axis=0) for sentence in transstate])
Xp.shape,Xs.shape,Xf.shape,Xt.shape

((4919, 768), (4919, 768), (4919, 768), (4919, 768))

In [ ]:
y=train_full['Etiqueta 2'].apply(categoricallabelAll)
y=y.to_numpy()
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6, 7])

In [ ]:
from sklearn.model_selection import train_test_split

Xp_train,Xp_test,Xcode_train,Xcode_test,Xs_train,Xs_test,Xt_train,Xt_test,Xf_train,Xf_test,y_train,y_test=train_test_split(Xp,Xcode,Xs,Xt,Xf,y,test_size=0.2,random_state=2023, stratify=y)

In [ ]:
Xp_train.shape,Xp_test.shape,Xcode_train.shape, Xcode_test.shape, Xs_train.shape,Xs_test.shape,Xt_train.shape,Xt_test.shape,Xf_train.shape,Xf_test.shape,y_train.shape,y_test.shape

((3935, 768),
 (984, 768),
 (3935, 768),
 (984, 768),
 (3935, 768),
 (984, 768),
 (3935, 768),
 (984, 768),
 (3935, 768),
 (984, 768),
 (3935,),
 (984,))

# Tradicional Machine Learning


In [ ]:
X_train=np.concatenate((Xp_train,Xcode_train,Xs_train,Xt_train,Xf_train),axis=1)
X_test=np.concatenate((Xp_test,Xcode_test,Xs_test,Xt_test,Xf_test),axis=1)

In [ ]:
X_train.shape,X_test.shape

((3935, 3840), (984, 3840))

In [ ]:
#random search
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import matthews_corrcoef
#XGBoost
from xgboost import XGBClassifier
#pipeline
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier


In [ ]:
from sklearn.metrics import matthews_corrcoef, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

def calculate_mcc_multiclass(y_true, y_pred_probs):
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    # Convertir etiquetas verdaderas one-hot a etiquetas enteras si es necesario
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    return matthews_corrcoef(y_true, y_pred_labels)

def calculate_auc_pr_multiclass(y_true, y_pred_probs, average='macro'):
    n_classes = y_pred_probs.shape[1]
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    auc_pr_list = []
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs[:, i])
        auc_pr_list.append(auc(recall, precision))

    if average == 'macro':
        return np.mean(auc_pr_list)
    elif average == 'weighted':
        class_counts = y_true_bin.sum(axis=0)
        return np.average(auc_pr_list, weights=class_counts)
    else:
        return auc_pr_list


In [ ]:
#pipelines
psvc=Pipeline([
    ('scaler',StandardScaler()),
    ('classifier',SVC(probability=True))
])
pmlp=Pipeline([
    ('scaler',StandardScaler()),
    ('classifier',MLPClassifier())
])

In [ ]:
20*20*3

1200

In [ ]:
X_train2,X_test2,y_train2,y_test2=train_test_split(X_test,y_test,test_size=0.2,random_state=2023, stratify=y_test)

In [ ]:
#svc search
param_grid_svc={
    'classifier__C':np.logspace(-5,5,20),
    'classifier__gamma':np.logspace(-5,5,20),
    'classifier__kernel':['rbf']
}
#search
svc_search=RandomizedSearchCV(psvc,param_grid_svc,cv=5,n_jobs=-1,verbose=1,n_iter=10)
svc_search.fit(X_train2,y_train2)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                             ('classifier',
                                              SVC(probability=True))]),
                   n_jobs=-1,
                   param_distributions={'classifier__C': array([1.00000000e-05, 3.35981829e-05, 1.12883789e-04, 3.79269019e-04,
       1.27427499e-03, 4.28133240e-03, 1.43844989e-02, 4.83293024e-02,
       1.62377674e-01, 5.45559478e-01, 1.83298071e+00, 6.15848211e+00...
                                        'classifier__gamma': array([1.00000000e-05, 3.35981829e-05, 1.12883789e-04, 3.79269019e-04,
       1.27427499e-03, 4.28133240e-03, 1.43844989e-02, 4.83293024e-02,
       1.62377674e-01, 5.45559478e-01, 1.83298071e+00, 6.15848211e+00,
       2.06913808e+01, 6.95192796e+01, 2.33572147e+02, 7.84759970e+02,
       2.63665090e+03, 8.85866790e+03, 2.97635144e+04, 1.00000000e+05]),
                                        'classifier__kernel': ['rbf']},
                   verbose=1)

In [ ]:
print(svc_search.best_params_)
print(svc_search.best_score_)
print("Test data:",svc_search.score(X_test,y_test))
#Metrics
y_pred=svc_search.predict(X_test)
print(classification_report(y_test,y_pred))
##MCC
mcc = matthews_corrcoef(y_test, y_pred)
print("MCC:", mcc)
##AUC
auc_pr = calculate_auc_pr_multiclass(y_test, svc_search.predict_proba(X_test))
print("AUC-PR:", auc_pr)

{'classifier__kernel': 'rbf', 'classifier__gamma': np.float64(0.000379269019073225), 'classifier__C': np.float64(233.57214690901213)}
0.6226477465129404
Test data: 0.9197154471544715
              precision    recall  f1-score   support

           0       0.91      0.93      0.92       292
           1       0.92      0.92      0.92       198
           2       0.95      0.94      0.95       197
           3       0.89      0.89      0.89       197
           4       0.96      0.92      0.94        26
           5       0.95      0.84      0.89        25
           6       0.92      0.86      0.89        28
           7       0.87      0.95      0.91        21

    accuracy                           0.92       984
   macro avg       0.92      0.91      0.91       984
weighted avg       0.92      0.92      0.92       984

MCC: 0.8980722716071247
AUC-PR: 0.9379798896308801


In [ ]:
#mlp search
param_grid_mlp={
    'classifier__hidden_layer_sizes': [(8),(10),(16),(32),(8,8),(10,10),(16,16),(32,32),(8,8,8),(10,10,10),(16,16,16),(32,32,32),(100,100,100)],
    'classifier__learning_rate': ['invscaling', 'adaptive'],
}
#search
mlp_search=RandomizedSearchCV(pmlp,param_grid_mlp,cv=5,n_jobs=-1,verbose=1,n_iter=10)
mlp_search.fit(X_train2,y_train2)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                             ('classifier', MLPClassifier())]),
                   n_jobs=-1,
                   param_distributions={'classifier__hidden_layer_sizes': [8,
                                                                           10,
                                                                           16,
                                                                           32,
                                                                           (8,
                                                                            8),
                                                                           (10,
                                                                            10),
                                                                           (16,
                                                                            16),
                                                                           (32,
                                                                            32),
                                                                           (8,
                                                                            8,
                                                                            8),
                                                                           (10,
                                                                            10,
                                                                            10),
                                                                           (16,
                                                                            16,
                                                                            16),
                                                                           (32,
                                                                            32,
                                                                            32),
                                                                           (100,
                                                                            100,
                                                                            100)],
                                        'classifier__learning_rate': ['invscaling',
                                                                      'adaptive']},
                   verbose=1)

In [ ]:
print(mlp_search.best_params_)
print(mlp_search.best_score_)
print("Test data:",mlp_search.score(X_test,y_test))
#Metrics
y_pred=mlp_search.predict(X_test)
print(classification_report(y_test,y_pred))
##MCC
mcc = matthews_corrcoef(y_test, y_pred)
print("MCC:", mcc)
##AUC
auc_pr = calculate_auc_pr_multiclass(y_test, mlp_search.predict_proba(X_test))
print("AUC-PR:", auc_pr)


{'classifier__learning_rate': 'adaptive', 'classifier__hidden_layer_sizes': (100, 100, 100)}
0.6252197049101024
Test data: 0.9065040650406504
              precision    recall  f1-score   support

           0       0.92      0.89      0.91       292
           1       0.88      0.93      0.90       198
           2       0.93      0.94      0.94       197
           3       0.88      0.89      0.89       197
           4       0.96      0.92      0.94        26
           5       0.88      0.84      0.86        25
           6       0.89      0.86      0.87        28
           7       0.95      0.90      0.93        21

    accuracy                           0.91       984
   macro avg       0.91      0.90      0.90       984
weighted avg       0.91      0.91      0.91       984

MCC: 0.8816356896105288
AUC-PR: 0.9269972368500325


In [ ]:
#search random forest
param_grid_rf={
    'n_estimators':[10,20,30,40,50,60,70,80,90,100],
    'max_depth':[5,10,15]
}
#search
rf_search=RandomizedSearchCV(RandomForestClassifier(),param_grid_rf,cv=5,n_jobs=-1,verbose=1,n_iter=10)
rf_search.fit(X_train,y_train)


Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(), n_jobs=-1,
                   param_distributions={'max_depth': [5, 10, 15],
                                        'n_estimators': [10, 20, 30, 40, 50, 60,
                                                         70, 80, 90, 100]},
                   verbose=1)

In [ ]:
print(rf_search.best_params_)
print(rf_search.best_score_)
print("Test data:",rf_search.score(X_test,y_test))
# Metrics
y_pred=rf_search.predict(X_test)
print(classification_report(y_test,y_pred))
##MCC
mcc = matthews_corrcoef(y_test, y_pred)
print("MCC:", mcc)
##AUC
auc_pr = calculate_auc_pr_multiclass(y_test, rf_search.predict_proba(X_test))
print("AUC-PR:", auc_pr)


{'n_estimators': 70, 'max_depth': 15}
0.9047013977128335
Test data: 0.9400406504065041
              precision    recall  f1-score   support

           0       0.92      0.92      0.92       292
           1       0.97      0.95      0.96       198
           2       0.97      0.96      0.97       197
           3       0.89      0.94      0.91       197
           4       1.00      0.96      0.98        26
           5       1.00      0.76      0.86        25
           6       1.00      0.93      0.96        28
           7       1.00      1.00      1.00        21

    accuracy                           0.94       984
   macro avg       0.97      0.93      0.95       984
weighted avg       0.94      0.94      0.94       984

MCC: 0.9239503482012701
AUC-PR: 0.9653852016061948


In [ ]:
param_grid_knn={
    'n_neighbors':[3,5,7,9,11,13,15,17,19,21],
    'weights':['uniform','distance'],
    'p':[1,2]
}
knn_search=RandomizedSearchCV(KNeighborsClassifier(),param_grid_knn,cv=5,n_jobs=-1,verbose=1,n_iter=10)
knn_search.fit(X_train,y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5, estimator=KNeighborsClassifier(), n_jobs=-1,
                   param_distributions={'n_neighbors': [3, 5, 7, 9, 11, 13, 15,
                                                        17, 19, 21],
                                        'p': [1, 2],
                                        'weights': ['uniform', 'distance']},
                   verbose=1)

In [51]:
print(knn_search.best_params_)
print(knn_search.best_score_)
print("Test data:",knn_search.score(X_test,y_test))
# Metrics
y_pred=knn_search.predict(X_test)
print(classification_report(y_test,y_pred))
##MCC
mcc = matthews_corrcoef(y_test, y_pred)
print("MCC:", mcc)
##AUC
auc_pr = calculate_auc_pr_multiclass(y_test, knn_search.predict_proba(X_test))
print("AUC-PR:", auc_pr)

{'weights': 'distance', 'p': 1, 'n_neighbors': 3}
0.8775095298602288
Test data: 0.9004065040650406
              precision    recall  f1-score   support

           0       0.90      0.83      0.86       292
           1       0.93      0.93      0.93       198
           2       0.94      0.95      0.95       197
           3       0.83      0.91      0.87       197
           4       0.93      0.96      0.94        26
           5       0.79      0.76      0.78        25
           6       1.00      0.93      0.96        28
           7       1.00      1.00      1.00        21

    accuracy                           0.90       984
   macro avg       0.91      0.91      0.91       984
weighted avg       0.90      0.90      0.90       984

MCC: 0.8745241037893227
AUC-PR: 0.9515437658415379


# KNN

In [52]:
knn=KNeighborsClassifier(n_neighbors=3,p=1,weights='distance')
knn.fit(X_train,y_train)

KNeighborsClassifier(n_neighbors=3, p=1, weights='distance')

In [53]:
knn.score(X_test,y_test)
# Metrics
y_pred=knn.predict(X_test)
print(classification_report(y_test,y_pred))
##MCC
mcc = matthews_corrcoef(y_test, y_pred)
print("MCC:", mcc)
##AUC
auc_pr = calculate_auc_pr_multiclass(y_test, knn.predict_proba(X_test))
print("AUC-PR:", auc_pr)

              precision    recall  f1-score   support

           0       0.90      0.83      0.86       292
           1       0.93      0.93      0.93       198
           2       0.94      0.95      0.95       197
           3       0.83      0.91      0.87       197
           4       0.93      0.96      0.94        26
           5       0.79      0.76      0.78        25
           6       1.00      0.93      0.96        28
           7       1.00      1.00      1.00        21

    accuracy                           0.90       984
   macro avg       0.91      0.91      0.91       984
weighted avg       0.90      0.90      0.90       984

MCC: 0.8745241037893227
AUC-PR: 0.9515437658415379


# SVM

In [54]:
#{'classifier__kernel': 'rbf', 'classifier__gamma': np.float64(0.000379269019073225), 'classifier__C': np.float64(233.57214690901213)}
svm_pipe=Pipeline([
    ('scaler',StandardScaler()),
    ('classifier',SVC(kernel='rbf',gamma=0.000379269019073225,C=233.57214690901213,probability=True))
])
svm_pipe.fit(X_train,y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 SVC(C=233.57214690901213, gamma=0.000379269019073225,
                     probability=True))])

In [55]:
print(svm_pipe.score(X_test,y_test))
#Metrics
y_pred=svm_pipe.predict(X_test)
print(classification_report(y_test,y_pred))
#MCC
mcc = matthews_corrcoef(y_test, y_pred)
print("MCC:", mcc)
#AUC
auc_pr = calculate_auc_pr_multiclass(y_test, svm_pipe.predict_proba(X_test))
print("AUC-PR:", auc_pr)

0.9390243902439024
              precision    recall  f1-score   support

           0       0.93      0.91      0.92       292
           1       0.93      0.96      0.95       198
           2       0.97      0.96      0.97       197
           3       0.92      0.94      0.93       197
           4       1.00      0.96      0.98        26
           5       0.95      0.72      0.82        25
           6       0.96      0.93      0.95        28
           7       1.00      1.00      1.00        21

    accuracy                           0.94       984
   macro avg       0.96      0.92      0.94       984
weighted avg       0.94      0.94      0.94       984

MCC: 0.9226708079035775
AUC-PR: 0.9592558884870259


#MLP

In [56]:
mlp_pipe=Pipeline([
    ('scaler',StandardScaler()),
    ('classifier',MLPClassifier(hidden_layer_sizes=(100,100,100),learning_rate='adaptive'))
])
mlp_pipe.fit(X_train,y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 MLPClassifier(hidden_layer_sizes=(100, 100, 100),
                               learning_rate='adaptive'))])

In [58]:
print(mlp_pipe.score(X_test,y_test))
#Metrics
y_pred=mlp_pipe.predict(X_test)
print(classification_report(y_test,y_pred))
#MCC
mcc = matthews_corrcoef(y_test, y_pred)
print("MCC:", mcc)
#AUC
auc_pr = calculate_auc_pr_multiclass(y_test, mlp_pipe.predict_proba(X_test))
print("AUC-PR:", auc_pr)

0.9247967479674797
              precision    recall  f1-score   support

           0       0.90      0.89      0.90       292
           1       0.94      0.95      0.95       198
           2       0.97      0.97      0.97       197
           3       0.89      0.89      0.89       197
           4       1.00      0.96      0.98        26
           5       0.86      0.76      0.81        25
           6       0.97      1.00      0.98        28
           7       0.91      1.00      0.95        21

    accuracy                           0.92       984
   macro avg       0.93      0.93      0.93       984
weighted avg       0.92      0.92      0.92       984

MCC: 0.9046790505769173
AUC-PR: 0.9624605805424259


# Random Forest

In [59]:
rf=RandomForestClassifier(n_estimators=80,max_depth=15)
rf.fit(X_train,y_train)


RandomForestClassifier(max_depth=15, n_estimators=80)

In [60]:
print(rf.score(X_test,y_test))
#Metrics
y_pred=rf.predict(X_test)
print(classification_report(y_test,y_pred))
#MCC
mcc = matthews_corrcoef(y_test, y_pred)
print("MCC:", mcc)
#AUC
auc_pr = calculate_auc_pr_multiclass(y_test, rf.predict_proba(X_test))
print("AUC-PR:", auc_pr)


0.931910569105691
              precision    recall  f1-score   support

           0       0.91      0.92      0.91       292
           1       0.95      0.94      0.95       198
           2       0.97      0.96      0.97       197
           3       0.88      0.92      0.90       197
           4       1.00      0.96      0.98        26
           5       1.00      0.76      0.86        25
           6       1.00      0.89      0.94        28
           7       1.00      1.00      1.00        21

    accuracy                           0.93       984
   macro avg       0.96      0.92      0.94       984
weighted avg       0.93      0.93      0.93       984

MCC: 0.9135222104868382
AUC-PR: 0.9684408352726774
